# <center> <img src="../img/ITESOLogo.png" alt="ITESO" width="480" height="130"> </center>
# <center> **Departamento de Electrónica, Sistemas e Informática** </center>
---
## <center> **Big Data** </center>
---
### <center> **Spring 2026** </center>
---
### <center> **Examples on Machine Learning: Alternating Least Squares (ALS)** </center>
---
##### Luis Guillermo Rivera Stephens
**Profesor**: Pablo Camarillo Ramirez

# Create SparkSession

In [2]:
from pcamarillor.spark_utils import SparkUtils

su = SparkUtils("ML: ALS", 
                "spark://spark-master:7077")
su.spark

# Example 1: Songs recommednation

In [3]:
# Sample user-song interaction data
data = [(1, 1, 4),
        (1, 2, 5),
        (1, 5, 5),
        (2, 2, 3),
        (2, 3, 4),
        (2, 4, 3),
        (3, 1, 2),
        (3, 3, 5),
        (3, 5, 1)]
  
# Define schema for the DataFrame
schema = SparkUtils.generate_schema([("user_id", "int"), ("song_id", "int"), ("rating", "int")])

# Create DataFrame for interactions
interactions_df = su.spark.createDataFrame(data, schema)
interactions_df.show()

+-------+-------+------+
|user_id|song_id|rating|
+-------+-------+------+
|      1|      1|     4|
|      1|      2|     5|
|      1|      5|     5|
|      2|      2|     3|
|      2|      3|     4|
|      2|      4|     3|
|      3|      1|     2|
|      3|      3|     5|
|      3|      5|     1|
+-------+-------+------+



In [4]:
print(f"Number of items o canciones (n):{interactions_df.groupBy('song_id').count().count()}")
print(f"Number of users (m):{interactions_df.groupBy('user_id').count().count()}")

Number of items o canciones (n):5
Number of users (m):3


In [5]:
from pyspark.ml.recommendation import ALS

als = ALS(
    userCol="user_id", 
    itemCol="song_id", 
    ratingCol="rating", 
    maxIter=10, 
    regParam=0.1, 
    rank=5, # Controls the dimensionality of the latent vector space for 
            # users and items.
    coldStartStrategy="drop"  # Avoids NaN predictions
)

In [6]:
model = als.fit(interactions_df)
print("Recommendation system generated successfully")

Recommendation system generated successfully


In [7]:
# Generate recommendations for each user
user_recommendations = model.recommendForAllUsers(numItems=3)

# Show recommendations
user_recommendations.show(truncate=False)

+-------+----------------------------------------------+
|user_id|recommendations                               |
+-------+----------------------------------------------+
|1      |[{2, 4.963741}, {5, 4.847486}, {1, 3.9368277}]|
|2      |[{3, 3.945299}, {2, 2.9739773}, {4, 2.907595}]|
|3      |[{3, 4.838227}, {4, 3.383236}, {2, 2.7535493}]|
+-------+----------------------------------------------+



In [8]:
songs = [
    (1, "song a"),
    (2, "song b"),
    (3, "song c"),
    (4, "song d"),
    (5, "song e")]

songs_schema = SparkUtils.generate_schema([("song_id", "int"), ("title", "string")])
songs_df = su.spark.createDataFrame(songs, songs_schema)

In [10]:
from pyspark.sql.functions import explode

# Explode recommendations for easier reading
recommendations = user_recommendations.select("user_id", explode("recommendations").alias("rec"))
recommendations = recommendations.join(songs_df, recommendations.rec.song_id == songs_df.song_id).select("user_id", "title", "rec.rating")

# Show user-song recommendations with titles
recommendations.show(truncate=False)

+-------+------+---------+
|user_id|title |rating   |
+-------+------+---------+
|1      |song b|4.963741 |
|1      |song e|4.847486 |
|1      |song a|3.9368277|
|2      |song c|3.945299 |
|2      |song b|2.9739773|
|2      |song d|2.907595 |
|3      |song c|4.838227 |
|3      |song d|3.383236 |
|3      |song b|2.7535493|
+-------+------+---------+



In [11]:
predictions = model.transform(interactions_df)
predictions.show(truncate=False)

+-------+-------+------+----------+
|user_id|song_id|rating|prediction|
+-------+-------+------+----------+
|1      |1      |4     |3.9368277 |
|1      |2      |5     |4.963741  |
|1      |5      |5     |4.847486  |
|2      |2      |3     |2.9739773 |
|3      |1      |2     |1.9615636 |
|3      |3      |5     |4.838227  |
|3      |5      |1     |1.042146  |
|2      |3      |4     |3.945299  |
|2      |4      |3     |2.907595  |
+-------+-------+------+----------+



In [12]:
# Evaluate the Recommendation System
from pyspark.ml.evaluation import RegressionEvaluator
# Set up evaluator to compute RMSE
evaluator = RegressionEvaluator(
    metricName="rmse", 
    labelCol="rating", 
    predictionCol="prediction"
)

# Calculate RMSE
rmse = evaluator.evaluate(predictions)
print(f"Root-mean-square error (RMSE) = {rmse}")

Root-mean-square error (RMSE) = 0.08831652956670598


# Lab 12: Building a Recommendation System with ALS 

In [15]:
movies_ratings_path = "/opt/spark/work-dir/data/ml/als/"

movies_ratings_schema = SparkUtils.generate_schema([("user_id", "int"), ("movie_id", "int"), ("rating", "int"),("timestamp", "int")])

# Source https://github.com/databricks/Spark-The-Definitive-Guide/blob/master/data/sample_movielens_ratings.txt
movies_ratings_df = su.spark.read \
                    .option("header", "false") \
                    .option("delimiter", "::") \
                    .schema(movies_ratings_schema) \
                    .csv(movies_ratings_path)

movies_ratings_df.printSchema()
movies_ratings_df.show(n=3)

root
 |-- user_id: integer (nullable = true)
 |-- movie_id: integer (nullable = true)
 |-- rating: integer (nullable = true)
 |-- timestamp: integer (nullable = true)

+-------+--------+------+----------+
|user_id|movie_id|rating| timestamp|
+-------+--------+------+----------+
|      0|       2|     3|1424380312|
|      0|       3|     1|1424380312|
|      0|       5|     2|1424380312|
+-------+--------+------+----------+
only showing top 3 rows


## Create & Train the ML Model

In [53]:
als = ALS(
    userCol="user_id", 
    itemCol="movie_id", 
    ratingCol="rating", 
    maxIter=20, 
    regParam=0.1, 
    rank=5, # Controls the dimensionality of the latent vector space for 
            # users and items.
    coldStartStrategy="drop"  # Avoids NaN predictions
)

model = als.fit(movies_ratings_df)

## Persist the model

## Predictions

In [54]:
predictions = model.transform(movies_ratings_df)
predictions.show(truncate=False)

+-------+--------+------+----------+----------+
|user_id|movie_id|rating|timestamp |prediction|
+-------+--------+------+----------+----------+
|22     |0       |1     |1424380312|0.95797   |
|22     |3       |2     |1424380312|1.6246145 |
|22     |5       |2     |1424380312|2.034913  |
|22     |6       |2     |1424380312|2.2904074 |
|22     |9       |1     |1424380312|1.5315897 |
|22     |10      |1     |1424380312|1.4318569 |
|22     |11      |1     |1424380312|1.2991562 |
|22     |13      |1     |1424380312|1.5953199 |
|22     |14      |1     |1424380312|1.3870223 |
|22     |16      |1     |1424380312|0.70125186|
|22     |18      |3     |1424380312|3.0294096 |
|22     |19      |1     |1424380312|1.4549844 |
|22     |22      |5     |1424380312|4.1091666 |
|22     |25      |1     |1424380312|0.9897522 |
|22     |26      |1     |1424380312|1.1388614 |
|22     |29      |3     |1424380312|3.2443485 |
|22     |30      |5     |1424380312|3.9998507 |
|22     |32      |4     |1424380312|3.18

## Test ML Model

In [55]:
evaluator = RegressionEvaluator(
    metricName="rmse", 
    labelCol="rating", 
    predictionCol="prediction"
)

# Calculate RMSE
rmse = evaluator.evaluate(predictions)
print(f"Root-mean-square error (RMSE) = {rmse}")

Root-mean-square error (RMSE) = 0.5694630155290662


In [ ]:
su.spark.stop()